# TaaS Studio — Hypergraph Explorer Browser E2E (Colab Runtime)

This notebook runs the real Studio API and frontend in headless Chromium using Playwright. It exercises authentication, real Core discovery, the projection UI, filtering, keyboard selection, the inspector and responsive layout. The demo capability is synthetic metadata only: no MCP server is contacted and no tool is executed.


In [ ]:
from pathlib import Path
import subprocess, sys, os

REPO = "https://github.com/msellamiTN/taas-provenance-discovery.git"
PIN = "2a63165d712082817660fb67b7f73159b907f8cb"
WORKDIR = Path("/content/taas-provenance-discovery")
if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--detach", PIN], check=True)
actual = subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"], text=True).strip()
assert actual == PIN, (actual, PIN)
print("Pinned source revision:", actual)
os.environ["TAAS_E2E_SCREENSHOT"] = "/content/hypergraph-explorer-browser-e2e.png"
%cd /content/taas-provenance-discovery


In [ ]:
!pip -q install -e '.[test,studio]' playwright
!playwright install --with-deps chromium
!python -m compileall -q src/taas tests/test_studio_browser_e2e.py


In [ ]:
!pytest -q tests/test_retrieval_degenerate.py tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py tests/test_studio_browser_e2e.py -s


In [ ]:
from IPython.display import display, Image
artifact = Path(os.environ["TAAS_E2E_SCREENSHOT"])
assert artifact.is_file() and artifact.stat().st_size > 0, "Browser screenshot artifact was not created"
display(Image(filename=str(artifact)))
print("Browser screenshot:", artifact)


## Expected validation boundary

- Chromium loads the actual Studio page served by the FastAPI application.
- The page authenticates with a test-only token and the real API.
- Discovery returns a Core-generated `taas.hypergraph-projection/v1` projection with `execution_performed=false`.
- Search, SVG keyboard selection, accessible list selection and inspector interactions work.
- The responsive graph/inspector layout stacks at a narrow viewport.
- No `/api/v1/mcp/*` browser request is made.

A passing run validates this pinned browser journey only. It is not production deployment, authorization, live MCP execution, cross-browser, screen-reader or human visual sign-off evidence. Preserve the Colab output as operator evidence; do not conflate it with GitHub Actions.